# Lab 01-E — Similaridade de cosseno

**Skill & GO · Agentic Engineering · Aula 01 — Fundamentos**

No Lab 01-C, cada texto virou um vetor (embedding). Agora vamos **comparar vetores**:

1. calcular a **similaridade de cosseno** entre dois vetores;
2. usar esse cálculo para descobrir quais textos são mais parecidos com uma consulta.

## 1. Preparar o ambiente

Instalamos a integração do LangChain com a OpenAI, usada para gerar os embeddings.

In [ ]:
%pip install -qU "langchain-openai>=1.6,<2"

Os embeddings são gerados pela API da OpenAI: no Colab, cadastre `OPENAI_API_KEY` no painel **🔑 Secrets** e habilite o acesso para este notebook.

In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    pass  # fora do Colab: usa a variável de ambiente
except Exception as erro:
    print(f"Não foi possível ler o segredo do Colab: {erro}")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Cole sua OPENAI_API_KEY: ")

print("Chave configurada ✅")

## 2. Similaridade de cosseno

A similaridade de cosseno mede o **ângulo entre dois vetores**, sem se importar com o tamanho deles:

$$\text{similaridade}(A, B) = \cos(\theta) = \frac{A \cdot B}{\|A\| \, \|B\|}$$

- **1**: apontam para a mesma direção (muito parecidos);
- **0**: são perpendiculares (sem relação);
- **-1**: apontam para direções opostas.

Vamos testar com vetores de 2 dimensões, fáceis de imaginar.

In [ ]:
import numpy as np


def similaridade_cosseno(a, b) -> float:
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


print("Mesma direção   [1, 2] e [2, 4]: ", round(similaridade_cosseno([1, 2], [2, 4]), 3))
print("Perpendiculares [1, 0] e [0, 1]: ", round(similaridade_cosseno([1, 0], [0, 1]), 3))
print("Opostos         [1, 0] e [-1, 0]:", round(similaridade_cosseno([1, 0], [-1, 0]), 3))

Repare no primeiro caso: `[2, 4]` é duas vezes maior que `[1, 2]`, mas a similaridade é **1**, porque o cosseno só olha a direção.

## 3. Comparando embeddings

Agora os vetores têm **1536 dimensões** — impossível de desenhar, mas a conta é a mesma.

Geramos o embedding de uma **consulta** e de alguns **textos**, calculamos a similaridade de cada texto com a consulta e ordenamos do mais parecido para o menos parecido. É assim que a etapa de recuperação do RAG escolhe os trechos mais relevantes.

In [ ]:
import pandas as pd
from langchain_openai import OpenAIEmbeddings

MODELO_EMBEDDING = "text-embedding-3-small"  # @param {type:"string"}
CONSULTA = "cupom de desconto aplicado duas vezes"  # @param {type:"string"}

TEXTOS = [
    "O desconto é somado em dobro quando o cliente volta para a etapa de pagamento",
    "Erro ao aplicar cupom no checkout da loja",
    "O rastreamento da entrega parou de atualizar",
    "A teleconsulta cai quando o paciente entra na sala",
    "Receita de bolo de cenoura com cobertura de chocolate",
]

embeddings = OpenAIEmbeddings(model=MODELO_EMBEDDING)
vetor_consulta = embeddings.embed_query(CONSULTA)
vetores_textos = embeddings.embed_documents(TEXTOS)

resultado = pd.DataFrame({
    "texto": TEXTOS,
    "similaridade": [similaridade_cosseno(vetor_consulta, vetor) for vetor in vetores_textos],
})
resultado.sort_values("similaridade", ascending=False).round(3)

Observe na tabela:

- os dois textos sobre cupom e desconto ficam no topo — inclusive o que **não usa** as palavras "cupom" nem "duas vezes", mas descreve o mesmo problema. Isso é **similaridade semântica**: compara significados, não palavras;
- textos de outros assuntos ficam no fim da lista;
- com embeddings reais, textos sem relação não chegam a -1 e textos parecidos dificilmente chegam a 1. O que importa é a **ordem**; escolher um valor mínimo de corte é um trabalho de calibração.

> 💡 **Experimente:** troque a `CONSULTA` (por exemplo, "entrega atrasada") e veja a ordem mudar.